# 서울시 따릉이(공공자전거) 이용량 × 날씨 데이터 분석

- 명세서: `데이터분석_및_시각화PJT_명세서.pdf` (데이터 분석 및 시각화, 4회차)
- 담당 범위: **F401 ~ F403** (본 노트북 작성자)
- 2차 담당(F404~F406): 신현준
- 상세 계획/협업 방식: [`PLAN.md`](../PLAN.md), [`docs/collaboration_guide.md`](../docs/collaboration_guide.md)
- 데이터 출처·수집 기록: [`docs/data_collection_log.md`](../docs/data_collection_log.md)


## F401. 분석 문제 및 데이터 선정

### 분석 주제
**기상 조건(기온·강수·바람·습도)이 서울시 공공자전거(따릉이)의 일별 이용량에 미치는 영향**

### 분석 질문
1. **(주질문)** 일별 평균기온·강수량·풍속·습도는 일별 대여건수와 어떤 상관관계를 보이는가?
2. 비(강수)가 있었던 날과 없었던 날의 대여건수 차이는 유의미한가?
3. 월별(계절별)로 이용량 패턴과 날씨 영향력이 달라지는가?
4. 2025년과 2026년 동일 기간(1~6월)을 비교했을 때 이용량 추세에 변화가 있는가?

질문 2~4는 F404(EDA/시각화) 단계에서 2차 담당자가 본격적으로 탐색하며, 본 노트북(F401~F403)에서는
이 질문에 **답할 수 있는 데이터 형태**를 만드는 것까지를 범위로 한다.

### 사용 데이터
| 데이터셋 | 설명 | 기간 |
|---|---|---|
| 서울특별시 공공자전거 일별 대여건수 | 서울시 전체 따릉이 일별 대여건수 합계 (3개 CSV: 25.1-6 / 25.7-12 / 26.1-6) | 2025-01-01 ~ 2026-06-30 |
| 기상청 ASOS 108(서울) 일자료 | 서울 종관기상관측소(지점번호 108)의 일별 기상 관측값 | 2025-01-01 ~ 2026-10-14 |

### 분석 단위
**일(day) 단위, 서울시 전체** — 따릉이 데이터가 대여소별이 아닌 서울시 합산 일별 집계로 제공되므로,
분석 단위를 자연스럽게 "일자(date)"로 맞추고 ASOS 서울 관측소(108) 일자료와 날짜 기준으로 1:1 결합한다.

### 주요 변수 (선정)
- **종속변수**: `rentals` (일별 대여건수)
- **독립변수(후보)**: `avgTa`(평균기온), `minTa`(최저기온), `maxTa`(최고기온), `sumRn`(일강수량),
  `avgWs`(평균풍속), `avgRhm`(평균습도)
- 그 외 ASOS 컬럼(지면 상태, 일조/일사, 적설 등)은 결측이 많거나 분석 질문과 관련성이 낮아 F403에서 제외 여부를 판단한다.

### 선정 근거
- 두 데이터 모두 **공공데이터(서울 열린데이터광장 / 기상자료개방포털)** 로 출처가 명확하고 **일 단위로 기간이 거의 일치**해 결합이 쉽다.
- 따릉이는 서울의 대표적인 생활 교통수단이라 **날씨 민감도**를 확인하는 것이 실용적인 인사이트(운영·재배치 전략 등)로 이어질 수 있다.
- 두 데이터 모두 결측·형식 이슈가 있어(아래 F401 데이터 확인에서 검증) **정제 연습(F403)에도 적합**하다.


In [1]:
import pandas as pd

pd.set_option('display.max_columns', 20)


### 데이터 불러오기 및 기초 확인 (F401 근거 검증용)

In [2]:
bike_files = [
    "../data/raw/서울특별시 공공자전거 일별 대여건수_25.1-6.csv",
    "../data/raw/서울특별시 공공자전거 일별 대여건수_25.7-12.csv",
    "../data/raw/서울특별시 공공자전거 일별 대여건수_26.1-6.csv",
]

# 원본 CSV는 EUC-KR(cp949) 인코딩으로 저장되어 있어 UTF-8로 읽으면 헤더가 깨진다.
bike = pd.concat(
    [pd.read_csv(f, encoding="cp949") for f in bike_files],
    ignore_index=True,
)
bike.columns = ["date", "rentals"]
bike["date"] = pd.to_datetime(bike["date"])

print(bike.shape)
bike.head()


(546, 2)


,date,rentals
0,2025-01-01,48645
1,2025-01-02,77718
2,2025-01-03,77053
3,2025-01-04,56597
4,2025-01-05,19712


In [3]:
print("기간:", bike["date"].min().date(), "~", bike["date"].max().date())
print("중복 날짜 수:", bike["date"].duplicated().sum())

full_range = pd.date_range(bike["date"].min(), bike["date"].max(), freq="D")
missing_dates = full_range.difference(bike["date"])
print("빠진 날짜 수:", len(missing_dates))

bike["rentals"].describe()


기간: 2025-01-01 ~ 2026-06-30
중복 날짜 수: 0
빠진 날짜 수: 0


count       546.000000
mean     101377.441392
std       42067.699373
min        9975.000000
25%       66669.000000
50%      106466.500000
75%      138007.750000
max      176011.000000
Name: rentals, dtype: float64

In [4]:
asos = pd.read_csv("../data/raw/asos_108_daily_20250101_20261014.csv", encoding="utf-8")
asos["tm"] = pd.to_datetime(asos["tm"])

print(asos.shape)
print("기간:", asos["tm"].min().date(), "~", asos["tm"].max().date())
print("관측 지점:", asos[["stnId", "stnNm"]].drop_duplicates().to_dict("records"))

key_cols = ["tm", "avgTa", "minTa", "maxTa", "sumRn", "avgWs", "avgRhm"]
asos[key_cols].head()


(652, 62)
기간: 2025-01-01 ~ 2026-10-14
관측 지점: [{'stnId': 108, 'stnNm': '서울'}]


,tm,avgTa,minTa,maxTa,sumRn,avgWs,avgRhm
0,2025-01-01,2.6,-2.5,8.9,NaN,2.2,64.3
1,2025-01-02,0.5,-2.9,6.0,NaN,2.1,48.8
2,2025-01-03,-1.1,-5.1,3.2,0.0,2.2,52.0
3,2025-01-04,-0.1,-5.4,5.7,NaN,1.7,49.6
4,2025-01-05,1.2,-0.7,2.5,6.7,2.5,85.0


In [5]:
asos[key_cols].isna().sum()


tm          0
avgTa       0
minTa       0
maxTa       0
sumRn     386
avgWs       0
avgRhm      0
dtype: int64

### F401 데이터 확인 결과 해석

- 따릉이 데이터는 3개 파일을 합쳐 **546행**, 2025-01-01~2026-06-30 **연속 일자**이며 중복·누락 날짜가 없다.
  일별 대여건수는 평균 약 101,377건, 최소 9,975건~최대 176,011건으로 계절/날씨에 따른 변동 폭이 크다는 점이
  관찰된다(해석: 겨울철 저조, 여름철 급증 가능성 — 자세한 패턴 분석은 F404에서 수행).
- ASOS 데이터는 **652행**으로 2026-10-14까지 포함되어 따릉이 기간(~2026-06-30)보다 길다 →
  병합 시 따릉이 기간 기준으로 inner join 하면 자동으로 범위가 맞춰진다(F403에서 처리).
- 핵심 기상 변수 중 `avgTa`/`minTa`/`maxTa`/`avgWs`/`avgRhm`은 결측이 없고, `sumRn`(일강수량)만
  386일 결측이다. 이는 센서 미관측이 아니라 **강수가 없었던 날 공란으로 기록되는 ASOS 관례**로 보이며,
  F403에서 0으로 대체할지 별도 "강수 여부" 플래그로 다룰지 근거와 함께 결정한다.
- 두 데이터 모두 날짜 기준 결합이 가능하고(F401에서 정한 분석 단위: 일/서울시 전체), 결측·인코딩 이슈가
  명확히 파악되어 F403(정제) 진행이 가능한 상태임을 확인했다.

**다음 단계**: F402(수집 기록은 `docs/data_collection_log.md`에 별도 정리 완료) → F403(인코딩 통일,
날짜 범위 정렬, 결측 처리, 병합 후 `data/processed/`에 저장).


## F402. 데이터 수집 및 출처 기록

상세 기록(출처 URL, 수집 방법, 수집 시점, 이용 조건 전문)은 [`docs/data_collection_log.md`](../docs/data_collection_log.md)에
표로 정리했다. 아래는 핵심 요약과, 두 원본 데이터를 **분석 가능한 형태(일관된 인코딩·컬럼명·날짜 타입)** 로
표준화해 `data/processed/`에 저장하는 과정이다.

| 데이터셋 | 출처 | 수집 방법 | 수집 시점 | 데이터 범위 | 이용 조건 |
|---|---|---|---|---|---|
| 따릉이 일별 대여건수 | 서울 열린데이터광장(data.seoul.go.kr) | CSV 직접 다운로드 | 2026-10-15 | 2025-01-01~2026-06-30 | 공공데이터, 공공누리(KOGL) 출처표시 조건 통상 적용(정확한 유형은 포털에서 재확인 필요) |
| ASOS 108(서울) 일자료 | 기상자료개방포털(data.kma.go.kr), 지상 > 종관기상관측(ASOS) | CSV 직접 다운로드 | 2026-10-15 | 2025-01-01~2026-10-14 | 공공데이터, 공공누리(KOGL) 출처표시 조건 통상 적용(정확한 유형은 포털에서 재확인 필요) |

> 주의: 따릉이 데이터셋의 정확한 공식 명칭/페이지는 검색으로 완전히 특정하지 못했다(서울 열린데이터광장에서
> "공공자전거"로 검색해 정식 데이터셋명을 재확인하는 것을 권장). 이용 조건도 포털의 라이선스 유형을
> 정확히 확인하기 전까지는 "통상 공공누리 조건 적용"으로 잠정 기록한다. (한계로 F405에서도 재언급)


In [6]:
import os

os.makedirs("../data/processed", exist_ok=True)

# F401에서 이미 cp949/UTF-8로 올바르게 읽은 bike, asos를 재사용해
# "분석 가능한 형태(일관 인코딩·컬럼명)"로 표준화하여 저장한다.
# (값 자체의 정제·결측 처리는 F403에서 수행하며, 여기서는 형식만 표준화한다.)

bike_out_path = "../data/processed/bike_rentals_raw_utf8.csv"
asos_out_path = "../data/processed/asos_108_raw_utf8.csv"

bike.to_csv(bike_out_path, index=False, encoding="utf-8")
asos.to_csv(asos_out_path, index=False, encoding="utf-8")

print("저장 완료:", bike_out_path, bike.shape)
print("저장 완료:", asos_out_path, asos.shape)


저장 완료: ../data/processed/bike_rentals_raw_utf8.csv (546, 2)
저장 완료: ../data/processed/asos_108_raw_utf8.csv (652, 62)


### F402 결과 요약

- 두 원본 데이터 모두 공공데이터 포털에서 CSV로 직접 받은 자료이며, 출처·수집 방법·시점·범위·이용 조건을
  `docs/data_collection_log.md`에 표로 기록했다(이 노트북에는 요약만 포함).
- 원본의 인코딩 차이(따릉이 CSV는 cp949, ASOS는 UTF-8)를 통일해 `data/processed/`에 UTF-8로 저장함으로써
  "분석 가능한 형태"로 확보했다(F402 요구사항 충족). 값 자체의 정제(결측 처리 등)는 F403에서 이어서 수행한다.
